00b - Benchmark CoAtNet Raptor on a T4
#
# A single CoAtNet reportedly scores LB 0.924 with no ensembling and no TTA,
# which is the largest available single-model jump. The published cost is
# 23.7 GB at batch 8 on a 4090; the question is what it costs at batch 1-2 on
# the T4 we would actually train on.
#
# Measured, not estimated: a previous backbone benchmark without gradient
# checkpointing reported 21-25 h/fold because it was measuring host-RAM
# thrashing rather than compute, and nearly cost us the DINOv3 encoder.
# fp16 rather than bf16 throughout: a T4 is sm_75 and has no bf16
# tensor cores, so bf16 there is emulated and slower.

In [ ]:
import sys, subprocess
from pathlib import Path

CODE = next(
    (p for pat in ("rsna-knee-code", "*/rsna-knee-code", "*/*/rsna-knee-code")
     for p in sorted(Path("/kaggle/input").glob(pat)) if (p / "src").is_dir()),
    None,
)
assert CODE is not None, "rsna-knee-code dataset is not attached to this notebook"
sys.path.insert(0, str(CODE / "src"))
sys.path.insert(0, str(CODE))
print("CODE", CODE)

# timm ships in the Kaggle image; the install is a fallback for images that lack
# it, and cannot run in the offline submission notebook.
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=False)
    import timm

import rsna_knee.config as cfg
print("RAW ", cfg.RAW, cfg.RAW.exists())
print("PREP", cfg.PREP, cfg.PREP.exists())
print("SIZE", cfg.SIZE, cfg.MOSAIC_EXT)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
# Raptor's own resolution and epoch count, at the batch sizes a 16 GB card allows.
for B in (1, 2, 4):
    !cd {CODE} && python scripts/bench_backbones.py --size 384 --batch {B} --epochs 16 \n        --grad-checkpointing --models coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k \n        coatnet_rmlp_1_rw2_224.sw_in12k_ft_in1k convnext_tiny.dinov3_lvd1689m
